# **ML Project**

In [1]:
import pandas as pd
import numpy as np
import re

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from scipy.sparse import hstack

## **1. Preprocessing**
We start by converting the raw TXT dataset into CSV for easier handling. 
We then normalize Arabic text to reduce variations in letters (e.g., "أ" → "ا") and remove elongations (like "مررررحبا" → "مرحبا"). 
Emojis are replaced with tokens representing sentiment, and negation words are prefixed with "NOT_" to capture their effect in sentiment analysis.

### Turn TXT into CSV

In [2]:
labels = {"POS", "NEG", "OBJ", "NEUTRAL"}
data = []

with open("Arabic-Tweets_Dataset.txt", "r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()
        if not line:
            continue

        parts = line.split()

        # Last token must be a label
        if parts[-1] in labels:
            label = parts[-1]
            text = " ".join(parts[:-1])
            data.append([text, label])
        else:
            # skip malformed lines safely
            print("Skipped:", line)

df = pd.DataFrame(data, columns=["text", "label"])
df.to_csv("Arabic-Tweets_Dataset.csv", index=False, encoding="utf-8-sig")
df["label"] = df["label"].replace({"NEUTRAL": "OBJ"})

### Arabic PreProcessing Functions

In [3]:
emoji_dict = {
    "😂": " EMO_POS ",
    "❤️": " EMO_POS ",
    "😡": " EMO_NEG ",
    "😢": " EMO_NEG "
}

negations = ["مش", "مو", "ما", "ليس", "لا", "لم", "لن", "بدون", "غير"]

def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)
    return text

def remove_elongation(text):
    return re.sub(r'(.)\1+', r'\1', text)

def replace_emojis(text):
    for emoji, token in emoji_dict.items():
        text = text.replace(emoji, token)
    return text

def handle_negation(text):
    for neg in negations:
        text = text.replace(neg, " NOT_")
    return text

def preprocess_text(text):
    text = normalize_arabic(text)
    text = remove_elongation(text)
    text = replace_emojis(text)
    text = handle_negation(text)
    return text

### Apply Preprocessing

In [4]:
df["clean_text"] = df["text"].astype(str).apply(preprocess_text)
df[["text", "clean_text"]].head()

,text,clean_text
0,بعد استقالة رئيس #المحكمة_الدستورية ننتظر استق...,بعد استقاله ريس #ا NOT_حكمه_الدستوريه نتظر است...
1,أهنئ الدكتور أحمد جمال الدين، القيادي بحزب مصر...,اهني الدكتور احمد ج NOT_ل الدين، القيادي بحزب ...
2,البرادعي يستقوى بامريكا مرةاخرى و يرسل عصام ال...,البرادعي يستقوي بامريكا مرهاخري و يرسل عصام ال...
3,#الحرية_والعدالة | شاهد الآن: #ليلة_الاتحادية ...,#الحريه_والعداله | شاهد ا NOT_ن: #ليله_ا NOT_ت...
4,الوالدة لو اقولها بخاطري حشيشة تضحك بس من اقول...,الوالده لو اقولها بخاطري حشيشه تضحك بس من اقول...


## **2. Feature Extraction**
We extract both:
1. **Text-based features** using TF-IDF (max 5000 features, unigrams and bigrams).
2. **Handcrafted features** specific to Arabic:
   - Negation count
   - Dialect words presence
   - Emoji features

These are combined using `hstack` into a single feature matrix `X`.


### Arabic-Specific Feature Functions

In [5]:
dialect_words = ["مش", "مو", "شو", "ليش", "هيك"]

def negation_count(text):
    return sum(text.count(neg) for neg in negations)

def emoji_features(text):
    return {
        "has_positive_emoji": int("EMO_POS" in text),
        "has_negative_emoji": int("EMO_NEG" in text)
    }

def dialect_feature(text):
    return int(any(word in text for word in dialect_words))


### Extract Features

In [6]:
df["neg_count"] = df["clean_text"].apply(negation_count)
df["dialect"] = df["clean_text"].apply(dialect_feature)

emoji_df = df["clean_text"].apply(emoji_features).apply(pd.Series)
df = pd.concat([df, emoji_df], axis=1)

df[["neg_count", "dialect", "has_positive_emoji", "has_negative_emoji"]].head()

,neg_count,dialect,has_positive_emoji,has_negative_emoji
0,0,0,0,0
1,0,0,0,0
2,0,0,0,0
3,0,0,0,0
4,0,0,0,0


### TF-IDF Representation

In [7]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(df["clean_text"])

### Combine Features

In [8]:
handcrafted = df[[
    "neg_count",
    "dialect",
    "has_positive_emoji",
    "has_negative_emoji"
]].values

X = hstack([X_tfidf, handcrafted])
y = df["label"]

## **3. Dataset Split**
We split the data into:
- 60% training
- 20% validation
- 20% testing

We use stratified splitting to maintain the class distribution across all sets.

### Split Dataset (60-20-20)

In [9]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

## **4. Model Training**

#### Naïve Bayes
We use MultinomialNB, which works well for discrete features like word counts or TF-IDF. 
- Alpha = 0.5 is used for Laplace smoothing to avoid zero probabilities.

#### Neural Network (MLP)
We use a feed-forward neural network:
- Hidden layers: (128, 64) → two layers with decreasing size for hierarchical feature abstraction.
- Learning rate: 0.001 → standard starting point.
- Max iterations: 100 → limited to avoid long training times. Early stopping can be added for better efficiency.

### Naïve Bayes

In [10]:
nb = MultinomialNB(alpha=0.5)
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

cm_nb = confusion_matrix(y_test, y_pred_nb, labels=["NEG","OBJ","POS"])

### Neural Network 

In [11]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    learning_rate_init=0.001,
    max_iter=100,
    random_state=42
)

mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)

## **5. Model Evaluation**
We evaluate models using three metrics:

- **Accuracy**: overall fraction of correct predictions.
- **Classification Report**: includes **precision**, **recall**, and **F1-score** per class.
- **Confusion Matrix**: raw counts of predicted vs actual labels.

### Evaluate Naive Bayes

In [12]:
# Accuracy
acc_nb = accuracy_score(y_test, y_pred_nb)
print(f"Naïve Bayes Accuracy: {acc_nb:.2f}")

# Classification report
print("Naïve Bayes Classification Report:")
print(classification_report(y_test, y_pred_nb, target_names=["NEG","OBJ","POS"]))

# Confusion matrix
cm_nb = confusion_matrix(y_test, y_pred_nb, labels=["NEG","OBJ","POS"])
print("Naïve Bayes Confusion Matrix:")
print(cm_nb)

Naïve Bayes Accuracy: 0.75
Naïve Bayes Classification Report:
              precision    recall  f1-score   support

         NEG       0.52      0.10      0.17       337
         OBJ       0.76      0.98      0.86      1505
         POS       0.43      0.02      0.04       160

    accuracy                           0.75      2002
   macro avg       0.57      0.37      0.35      2002
weighted avg       0.70      0.75      0.68      2002

Naïve Bayes Confusion Matrix:
[[  34  302    1]
 [  30 1472    3]
 [   1  156    3]]


### Evaluate Naive Bayes

In [13]:
# Accuracy
acc_mlp = accuracy_score(y_test, y_pred_mlp)
print(f"MLP Accuracy: {acc_mlp:.2f}")

# Classification report
print("MLP Classification Report:")
print(classification_report(y_test, y_pred_mlp, target_names=["NEG","OBJ","POS"]))

# Confusion matrix
cm_mlp = confusion_matrix(y_test, y_pred_mlp, labels=["NEG","OBJ","POS"])
print("MLP Confusion Matrix:")
print(cm_mlp)

MLP Accuracy: 0.67
MLP Classification Report:
              precision    recall  f1-score   support

         NEG       0.32      0.33      0.33       337
         OBJ       0.79      0.79      0.79      1505
         POS       0.22      0.19      0.21       160

    accuracy                           0.67      2002
   macro avg       0.44      0.44      0.44      2002
weighted avg       0.66      0.67      0.66      2002

MLP Confusion Matrix:
[[ 112  212   13]
 [ 220 1190   95]
 [  18  111   31]]
